# 🔬 Sarcasm-Aware Explainable Early Detection of Cyberbullying
## IEEE Research Paper — Implementation Notebook

**Proposed System: SACENet** (Sarcasm-Aware Cyberbullying Early-detection Network)

### Architecture
1. **LLM Sarcasm Engine** — Gemini / GPT-4o / Local Twitter-RoBERTa-Irony
2. **Twitter-RoBERTa Cyberbullying Classifier** — Fine-tuned transformer backbone
3. **Sarcasm-Aware Score Fusion** — Dynamic probability boost for covert toxic intent
4. **SHAP Explainability** — Token-level attribution for model transparency
5. **Early Detection** — First-K message window analysis

---

## 📦 Section 1: Installation & Setup

In [ ]:
# Install all required packages
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'])
print('✅ All packages installed')

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
warnings.filterwarnings('ignore')

# ── Project modules ───────────────────────────────────────────────────────
from config import *
from dataset import CyberbullyingDataset, TextPreprocessor
from sarcasm_engine import SarcasmEngine, SarcasmResult
from model import SACENet, DetectionResult
from visualise import (
    plot_shap_waterfall, plot_sarcasm_quadrant,
    plot_architecture_diagram, plot_platform_distribution,
    plot_token_wordcloud,
)

print('✅ All modules loaded')
print(f'   Platform: {sys.platform}')
import torch
print(f'   PyTorch : {torch.__version__}')
print(f'   CUDA    : {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')

## 🔑 Section 2: API Key Configuration (Optional — LLM)

In [ ]:
# ── Set your API keys here ────────────────────────────────────────────────
# Leave empty to use the local BERT sarcasm model (no API needed)

import os

# Option A: Gemini (recommended — free tier available)
GEMINI_KEY = ''   # ← paste your Gemini API key here

# Option B: OpenAI
OPENAI_KEY = ''   # ← paste your OpenAI API key here

if GEMINI_KEY:
    os.environ['GEMINI_API_KEY'] = GEMINI_KEY
    SARCASM_PROVIDER = 'gemini'
    print('✅ Using Gemini for sarcasm detection')
elif OPENAI_KEY:
    os.environ['OPENAI_API_KEY'] = OPENAI_KEY
    SARCASM_PROVIDER = 'openai'
    print('✅ Using OpenAI for sarcasm detection')
else:
    SARCASM_PROVIDER = 'local'
    print('ℹ️  Using LOCAL BERT sarcasm model (twitter-roberta-base-irony)')
    print('   (No API key needed — best for offline/reproducible research)')

## 📊 Section 3: Dataset Loading & Exploration

In [ ]:
# ── Build multi-platform corpus ───────────────────────────────────────────
print('Building dataset...')
ds = CyberbullyingDataset(
    use_sample_corpus=True,
    balance=True,
    augment=True,
)

stats = ds.get_stats()
print(f"""
📊 Dataset Statistics
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Total samples          : {stats['total_samples']}
  Cyberbullying          : {stats['cyberbullying_count']}
  Non-cyberbullying      : {stats['non_cyberbullying_count']}
  Sarcastic bully posts  : {stats['sarcastic_bully_count']}
  Avg. text length       : {stats['avg_text_length']:.1f} words
  Platforms              : {list(stats['platform_distribution'].keys())}
""")

# Platform distribution figure (Section 4.1 of paper)
plot_platform_distribution(stats)
plt.show()

In [ ]:
# ── Preview dataset ───────────────────────────────────────────────────────
train_df, val_df, test_df = ds.get_splits()
print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')
display(train_df.head(8))

## 🎭 Section 4: LLM Sarcasm Engine Demo

In [ ]:
# ── Initialise sarcasm engine ─────────────────────────────────────────────
sarcasm_engine = SarcasmEngine(
    provider=SARCASM_PROVIDER,
    use_cache=True,
    fallback_to_local=True,
)
print(f'✅ Sarcasm engine ready: {sarcasm_engine.primary_provider}')

In [ ]:
# ── Run sarcasm detection on examples ────────────────────────────────────
test_examples = [
    ("Oh wow, you're SO smart! Only took you three days to figure out 2+2.",    'Sarcastic bully'),
    ("You are absolute garbage, nobody likes you, just disappear.",              'Direct bully'),
    ("Great work on your project! You've worked so hard and it really shows.",   'Non-bully'),
    ("Sure, your code is AMAZING. That's why it crashes every single time.",     'Sarcastic bully'),
    ("Oh great, it's raining again. Just what Monday needed.",                   'Benign sarcasm'),
    ("Congratulations! You've officially reached a new level of stupidity.",     'Sarcastic bully'),
]

print('\n🎭 SARCASM ENGINE RESULTS')
print('='*75)
results_table = []
for text, category in test_examples:
    r = sarcasm_engine.detect(text)
    icon = '🎭' if r.is_sarcastic else '💬'
    print(f"\n{icon} [{category}]")
    print(f"   Text        : {text[:70]}")
    print(f"   Score       : {r.sarcasm_score:.3f} ({r.confidence})")
    print(f"   Is Sarcastic: {r.is_sarcastic}")
    print(f"   Indicators  : {', '.join(r.indicators) or 'None'}")
    if r.explanation:
        print(f"   Explanation : {r.explanation[:120]}")
    results_table.append({
        'Category': category,
        'Text': text[:50] + '...',
        'Sarcasm Score': round(r.sarcasm_score, 3),
        'Is Sarcastic': r.is_sarcastic,
        'Confidence': r.confidence,
    })

print('\n\n📋 Summary Table:')
display(pd.DataFrame(results_table))

## 🤖 Section 5: SACENet — Full Pipeline Demo

In [ ]:
# ── Initialise SACENet ────────────────────────────────────────────────────
print('Loading SACENet...')
model = SACENet(
    sarcasm_provider=SARCASM_PROVIDER,
    enable_shap=True,      # Set False to skip SHAP (faster)
)
print('✅ SACENet ready')

In [ ]:
# ── Single-text detection with full explanation ───────────────────────────
text_to_analyse = "Oh wow, you're SO smart! Only took you three days to figure out 2+2."

print(f'📝 Analysing: "{text_to_analyse}"\n')
result = model.detect(text_to_analyse, platform='twitter', explain=True)

print('\n' + '='*70)
print('  DETECTION RESULT')
print('='*70)
print(result.summary())

if result.sarcasm_result:
    s = result.sarcasm_result
    print(f"\n🎭 Sarcasm Analysis:")
    print(f"   Score      : {s.sarcasm_score:.3f} ({s.confidence} confidence)")
    print(f"   Indicators : {', '.join(s.indicators) or 'None'}")
    if s.explanation:
        print(f"   Explanation: {s.explanation}")

if result.explanation:
    e = result.explanation
    print(f"\n🔍 SHAP Explanation:")
    print(f"   {e.shap_narrative}")
    if e.sarcasm_narrative:
        print(f"   {e.sarcasm_narrative}")
    print(f"\n   Top influential tokens:")
    for tok in e.top_tokens[:8]:
        direction = '→ Bully' if tok.shap_value > 0 else '→ Benign'
        print(f"   {tok.token:20} {tok.shap_value:+.4f}  {direction}")

print(f"\n⏱️  Detection time: {result.detection_time_ms:.0f} ms")

In [ ]:
# ── SHAP Waterfall Visualisation ─────────────────────────────────────────
if result.explanation and result.explanation.top_tokens:
    tokens      = [t.token      for t in result.explanation.top_tokens]
    shap_values = [t.shap_value for t in result.explanation.top_tokens]
    base_val    = result.explanation.base_value

    plot_shap_waterfall(
        tokens      = tokens,
        shap_values = shap_values,
        base_value  = base_val,
        prediction  = result.cyberbullying_prob,
        text        = text_to_analyse,
        title       = 'SACENet — SHAP Token Attribution',
        save_name   = 'shap_example_1',
    )
    img = plt.imread(os.path.join(PLOTS_DIR, 'shap_example_1.png'))
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.imshow(img)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    print('✅ SHAP plot saved to ./plots/')
else:
    print('ℹ️  SHAP disabled or no explanation available. Set enable_shap=True.')

## ⚡ Section 6: Early Detection Demonstration

In [ ]:
# ── Early detection on a simulated conversation ───────────────────────────
conversation = [
    "Hey, saw your new profile picture.",
    "Oh wow, very brave to post that. Most people would be too embarrassed.",
    "You're SOOO confident for someone who looks like that. Truly impressive.",
    "I mean some people just don't have mirrors I guess haha.",
    "Must be so easy being you, not caring what anyone thinks.",
]

print('📨 Simulated Conversation (Escalating Sarcastic Bullying):')
for i, msg in enumerate(conversation, 1):
    print(f'  [{i}] {msg}')

print('\n' + '─'*60)
print('⚡ Early Detection Analysis (K=3 messages):')

early_result = model.early_detect_conversation(
    messages=conversation,
    platform='instagram',
    k=3,
    explain=False,
)

print(f"\n📊 Risk Trajectory (per message): {[round(p, 3) for p in early_result['risk_trajectory']]}")
print(f"🚨 Escalation Detected : {early_result['escalation_detected']}")
print(f"📝 Messages Analysed   : {early_result['messages_analysed']}")
er = early_result['early_result']
print(f"\n{er.summary()}")

# Plot risk trajectory
fig, ax = plt.subplots(figsize=(8, 4))
k_vals = list(range(1, len(early_result['risk_trajectory']) + 1))
ax.plot(k_vals, early_result['risk_trajectory'], 'o-', color='#C44E52', lw=2, ms=8)
ax.axhline(CYBERBULLYING_THRESHOLD, color='black', ls='--', lw=1.2, label=f'Threshold ({CYBERBULLYING_THRESHOLD})')
ax.fill_between(k_vals, early_result['risk_trajectory'], alpha=0.15, color='#C44E52')
ax.set_xlabel('Message Number', fontsize=12)
ax.set_ylabel('Cyberbullying Probability', fontsize=12)
ax.set_title('Risk Trajectory — Early Detection (Sarcastic Escalation)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'risk_trajectory.png'), dpi=200)
plt.show()

## 🏋️ Section 7: Model Fine-Tuning (Optional)

In [ ]:
# ── Fine-tune the RoBERTa classifier on your corpus ───────────────────────
# NOTE: Requires ~8GB VRAM. Reduce batch size if OOM.
# Skip this cell if you want to use the pretrained model directly.

RUN_TRAINING = False   # ← Set to True to fine-tune

if RUN_TRAINING:
    from train import SACENetTrainer

    print('Tokenising dataset for training...')
    hf_dataset = ds.get_hf_dataset(tokenizer_name=CYBERBULLYING_MODEL)

    trainer = SACENetTrainer(base_model=CYBERBULLYING_MODEL)
    train_results = trainer.train(hf_dataset)

    print('\n📊 Training Results:')
    print(json.dumps(train_results['test_metrics'], indent=2))

    # Load fine-tuned model into SACENet
    model = SACENet(finetuned_path=MODELS_DIR, sarcasm_provider=SARCASM_PROVIDER)
    print('✅ Fine-tuned model loaded!')
else:
    print('ℹ️  Skipping training. Set RUN_TRAINING = True to fine-tune.')

## 📈 Section 8: Evaluation & Benchmarking

In [ ]:
# ── Run full evaluation vs baselines ─────────────────────────────────────
from evaluate import run_full_evaluation

print('Running full evaluation pipeline...')
print('(This compares SACENet vs TF-IDF+SVM and RoBERTa without sarcasm)\n')

eval_results = run_full_evaluation(save_results=True)

# Display comparison table
print('\n📊 MODEL COMPARISON')
comp_df = pd.DataFrame(eval_results['model_comparison'])
metric_cols = ['model', 'accuracy', 'f1_binary', 'precision', 'recall']
if 'roc_auc' in comp_df.columns:
    metric_cols.append('roc_auc')

display(comp_df[metric_cols].round(4).set_index('model'))

print('\n✅ All plots saved to ./plots/')

In [ ]:
# ── Display generated plots ───────────────────────────────────────────────
import glob
from IPython.display import Image, display as ipy_display

plot_files = sorted(glob.glob(os.path.join(PLOTS_DIR, '*.png')))
print(f'Generated {len(plot_files)} plots:')
for pf in plot_files:
    print(f'  {os.path.basename(pf)}')
    ipy_display(Image(pf, width=700))

## 🏗️ Section 9: Architecture Diagram

In [ ]:
from IPython.display import Image
path = plot_architecture_diagram()
print(f'Architecture diagram saved: {path}')
Image(path, width=900)

## 🔍 Section 10: Sarcasm Quadrant Analysis

In [ ]:
# ── Run batch detection on test set for quadrant plot ─────────────────────
print('Running batch detection on test set...')
test_texts  = test_df['text'].tolist()[:100]   # limit for speed
test_labels = test_df['label'].tolist()[:100]

batch_results = model.batch_detect(test_texts, explain=False)

bully_probs    = [r.cyberbullying_prob for r in batch_results]
sarcasm_scores = [r.sarcasm_result.sarcasm_score if r.sarcasm_result else 0.0 for r in batch_results]

path = plot_sarcasm_quadrant(
    bully_probs, sarcasm_scores, test_labels
)
print(f'✅ Quadrant plot saved: {path}')
from IPython.display import Image
Image(path, width=700)

## 📄 Section 11: Export Results for Paper

All results are saved in `./results/eval_results.json` and plots in `./plots/`.
Use these directly in your LaTeX paper.

In [ ]:
# ── Print final result table (copy-paste into LaTeX) ─────────────────────
results_path = os.path.join(RESULTS_DIR, 'eval_results.json')
if os.path.exists(results_path):
    with open(results_path) as f:
        saved = json.load(f)

    print('📋 LaTeX-Ready Results Table:\n')
    print('\\begin{table}[h]')
    print('\\centering')
    print('\\caption{Cyberbullying Detection Performance Comparison}')
    print('\\label{tab:results}')
    print('\\begin{tabular}{lcccc}')
    print('\\hline')
    print('Model & Accuracy & F1 & Precision & Recall \\\\')
    print('\\hline')
    for m in saved.get('model_comparison', []):
        print(f"{m['model']} & {m['accuracy']:.4f} & {m['f1_binary']:.4f} & {m['precision']:.4f} & {m['recall']:.4f} \\\\")
    print('\\hline')
    print('\\end{tabular}')
    print('\\end{table}')
else:
    print('ℹ️  Run Section 8 (Evaluation) first to generate results.')

---
## 📚 References

1. Liu, Y. et al. (2019). RoBERTa: A Robustly Optimized BERT Pretraining Approach. *arXiv:1907.11692*
2. Basile, V. et al. (2019). SemEval-2019 Task 5: HatEval. *ACL Anthology*
3. Lundberg, S.M. & Lee, S.-I. (2017). A Unified Approach to Interpreting Model Predictions (SHAP). *NeurIPS*
4. Founta, A.M. et al. (2018). Large Scale Crowdsourcing and Characterization of Twitter Abusive Behavior. *ICWSM*
5. Barbieri, F. et al. (2020). TweetEval: Unified Benchmark for Tweet Classification. *EMNLP Findings*
6. Google DeepMind. (2024). Gemini 1.5 Flash: Efficient Multimodal AI. *Technical Report*

---
*IEEE Research Paper — SACENet v1.0*